In [1]:
# Measure the time it takes every cell to run
%pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.2 MB/s eta 0:00:00
time: 264 µs (started: 2026-03-29 12:31:01 +00:00)


In [2]:
# Set up idc-index, used to make clinical table parameters human-readable
%pip install idc-index
from idc_index import IDCClient
c = IDCClient()
c.fetch_index('clinical_index')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 47.1 MB/s eta 0:00:00
time: 24.6 s (started: 2026-03-29 12:31:01 +00:00)


In [3]:
# Set up BigQuery, used to construct patient cohort of interest
import os
my_ProjectID = "nlst-radiomics"
os.environ["GCP_PROJECT_ID"] = my_ProjectID

from google.colab import auth, files
auth.authenticate_user()

from google.cloud import bigquery
bq_client = bigquery.Client(my_ProjectID)

time: 19.7 s (started: 2026-03-29 12:31:25 +00:00)


In [4]:
# First, what are the possible segmentation types?
# Note: I checked that 'AIMI lung and nodule AI segmentation' contains 'Lung' and 'Nodule' segment labels

seg_query = """
SELECT SeriesDescription
FROM `bigquery-public-data.idc_v23.dicom_all`
WHERE
  collection_id = 'nlst'
  AND Modality = 'SEG'
"""

seg_result = bq_client.query(seg_query)
seg_metadata_df = seg_result.result().to_dataframe()

print(f"\n--- Unique Series Descriptions ---")
display(seg_metadata_df['SeriesDescription'].value_counts().head(20))

print(f"\n--- Unique Series Descriptions Containing 'lung and nodule' ---")
lung_nodule_segmentation_types = seg_metadata_df[seg_metadata_df['SeriesDescription'].str.contains('lung and nodule', case=False)]['SeriesDescription'].unique().tolist()
display(lung_nodule_segmentation_types)


--- Unique Series Descriptions ---


,count
SeriesDescription,
TotalSegmentator(v1.5.6) Segmentation of Series 2,47987
TotalSegmentator(v1.5.6) Segmentation of Series 3,38198
TotalSegmentator(v1.5.6) Segmentation of Series 4,13362
TotalSegmentator(v1.5.6) Segmentation of Series 5,6180
TotalSegmentator(v1.5.6) Segmentation of Series 6,5377
TotalSegmentator(v1.5.6) Segmentation of Series 1,4449
TotalSegmentator(v1.5.6) Segmentation of Series 0,1132
AIMI lung and nodule AI segmentation,1042
3d_fullres-tta_nnU-Net_Segmentation,1039



--- Unique Series Descriptions Containing 'lung and nodule' ---


['AIMI lung and nodule radiologist 8 corrected segmentation',
 'AIMI lung and nodule radiologist 4 corrected segmentation',
 'AIMI lung and nodule radiologist 5 corrected segmentation',
 'AIMI lung and nodule AI segmentation']

time: 3.51 s (started: 2026-03-29 12:31:45 +00:00)


In [5]:
# How many CT/SEG pairs with lung and nodule segmentations are there?
# Note: If seg.SeriesDescription is not radiologist corrected, there are 1042 SEG files with lung and nodule segmentations

ct_seg_join_query = f"""
SELECT
  DISTINCT seg.SeriesInstanceUID AS SEG_SeriesInstanceUID,
  ct.PatientID,
  ct.SeriesInstanceUID AS CT_SeriesInstanceUID,
  seg.SeriesDescription AS SEG_SeriesDescription,
  seg.StudyDate,
  CASE
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '1999%' THEN 'T0'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2000%' THEN 'T1'
    WHEN FORMAT_DATE('%Y', seg.StudyDate) LIKE '2001%' THEN 'T2'
    ELSE 'Other'
  END AS StudyYear
FROM
  `bigquery-public-data.idc_v23.dicom_all` AS ct
JOIN
  `bigquery-public-data.idc_v23.dicom_all` AS seg
ON
  ct.collection_id = seg.collection_id
  AND ct.PatientID = seg.PatientID
WHERE
  ct.collection_id = 'nlst'
  AND ct.Modality = 'CT'
  AND seg.Modality = 'SEG'
  AND seg.SeriesDescription IN ({', '.join([f"'{s}'" for s in lung_nodule_segmentation_types])})
  AND EXISTS (
    SELECT 1
    FROM UNNEST(seg.ReferencedSeriesSequence) AS ref_seq
    WHERE ct.SeriesInstanceUID = ref_seq.SeriesInstanceUID
  )
"""

ct_seg_join_result = bq_client.query(ct_seg_join_query)
ct_seg_join_df = ct_seg_join_result.result().to_dataframe()

print(f"Number of distinct patients with a CT/SEG pair: {len(ct_seg_join_df['PatientID'].unique())}")
print(f"Number of unique CT series: {len(ct_seg_join_df['CT_SeriesInstanceUID'].unique())}")
print(f"Number of CT/SEG pairs: {ct_seg_join_df.shape[0]}")
print(f"Number of distinct study dates: {len(ct_seg_join_df['StudyDate'].unique())}")

Number of distinct patients with a CT/SEG pair: 572
Number of unique CT series: 1042
Number of CT/SEG pairs: 1144
Number of distinct study dates: 3
time: 1.99 s (started: 2026-03-29 12:31:49 +00:00)


In [6]:
# Two SEG series can correspond to one CT series because of radiologist corrected segmentations. "Keep" the CT/SEG pair with the radiologist corrected segmentation and throw out the other.

priority_segmentation_type = 'AIMI lung and nodule AI segmentation'

# Create a priority column: 0 for non-AI, 1 for AI
ct_seg_join_df['priority'] = ct_seg_join_df['SEG_SeriesDescription'].apply(lambda x: 1 if x == priority_segmentation_type else 0)

# Sort by CT_SeriesInstanceUID and then by priority (0 comes before 1, so non-AI is prioritized)
ct_seg_join_df_filtered = ct_seg_join_df.sort_values(by=['CT_SeriesInstanceUID', 'priority'])

# Drop duplicates, keeping the first (which will be the non-AI one if present, otherwise the first AI one)
ct_seg_join_df_one_to_one = ct_seg_join_df_filtered.drop_duplicates(subset='CT_SeriesInstanceUID', keep='first')

print(f"\nNumber of CT/SEG pairs after filtering: {ct_seg_join_df_one_to_one.shape[0]}")
display(ct_seg_join_df_one_to_one.head())


Number of CT/SEG pairs after filtering: 1042


,SEG_SeriesInstanceUID,PatientID,CT_SeriesInstanceUID,SEG_SeriesDescription,StudyDate,StudyYear,priority
967,1.2.276.0.7230010.3.1.3.17436516.3112497.17206...,107002,1.2.840.113654.2.55.10057224705584082952969157...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
306,1.2.276.0.7230010.3.1.3.17436516.3112767.17206...,107030,1.2.840.113654.2.55.10087518978221069034420730...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
943,1.2.276.0.7230010.3.1.3.17436516.3182127.17206...,126955,1.2.840.113654.2.55.10212803674068251180698456...,AIMI lung and nodule AI segmentation,2000-01-02,T1,1
1131,1.2.276.0.7230010.3.1.3.17436516.3103212.17206...,104999,1.2.840.113654.2.55.10222078521568314836955069...,AIMI lung and nodule AI segmentation,1999-01-02,T0,1
414,1.2.276.0.7230010.3.1.3.17436516.3180509.17206...,126446,1.2.840.113654.2.55.10223655654547815709262155...,AIMI lung and nodule radiologist 5 corrected s...,1999-01-02,T0,0


time: 135 ms (started: 2026-03-29 12:31:51 +00:00)


In [7]:
# Download CT/SEG pairs for 5 distinct patients using idc_index

# First, ensure the download directory is set on the Colab VM
download_dir_on_colab_vm = "/content/malignant_trial"
if not os.path.exists(download_dir_on_colab_vm):
    os.makedirs(download_dir_on_colab_vm)

# Your existing code to get series IDs
five_malignant_patientIDs = ct_seg_join_df_one_to_one['PatientID'].unique().tolist()[:5]
CT_SEG_seriesIDs = ct_seg_join_df_one_to_one[ct_seg_join_df_one_to_one['PatientID'].isin(five_malignant_patientIDs)][['CT_SeriesInstanceUID', 'SEG_SeriesInstanceUID']].values.flatten().tolist()

print(f"Downloading {len(CT_SEG_seriesIDs)} series for {len(five_malignant_patientIDs)} patients to Colab VM at {download_dir_on_colab_vm}...")
c.download_from_selection(seriesInstanceUID=CT_SEG_seriesIDs, downloadDir=download_dir_on_colab_vm)
print("Download to Colab VM complete.")

Download to Colab VM complete.
time: 11.4 s (started: 2026-03-29 12:31:51 +00:00)


In [8]:
# Compress the downloaded directory into a zip file
#zip_filename = 'malignant_trial_data.zip'
#!zip -r "{zip_filename}" "{download_dir_on_colab_vm}"

# Trigger download to your local machine
#print(f"\nTriggering download of '{zip_filename}' to your local machine...")
#files.download(zip_filename)
#print("Download initiated. Please check your browser's downloads.")

time: 944 µs (started: 2026-03-29 12:32:02 +00:00)


In [9]:
# first figure out how to convert CT file into a .nrrd file
%pip install SimpleITK
import SimpleITK as sitk
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 11.2 MB/s eta 0:00:00
time: 10.1 s (started: 2026-03-29 12:32:02 +00:00)


In [10]:
def convert_CT_to_nrrd(input_folder, output_folder):
  # assume input_folder and output_folder exist; case handling elsewhere
  reader = sitk.ImageSeriesReader()
  dicom_names = reader.GetGDCMSeriesFileNames(input_folder)
  reader.SetFileNames(dicom_names)
  ct_image = reader.Execute()
  sitk.WriteImage(ct_image, os.path.join(output_folder, 'CT.nrrd'))

time: 689 µs (started: 2026-03-29 12:32:12 +00:00)


In [11]:
%pip install dcmqi
import subprocess
import json
import glob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 MB 7.5 MB/s eta 0:00:00
time: 14.7 s (started: 2026-03-29 12:32:12 +00:00)


In [12]:
def convert_SEG_to_nrrd(input_file, output_folder, prefix="SEG"):
  command = ["segimage2itkimage", "--inputDICOM", input_file, "--outputDirectory", output_folder, "--outputType", "nrrd", "--prefix", prefix]
  try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    return True
  except subprocess.CalledProcessError as e:
    print(f"Conversion of SEG to .nrrd failed: {e}")
    return False

time: 822 µs (started: 2026-03-29 12:32:27 +00:00)


In [13]:
%pip install pydicom
import pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 21.3 MB/s eta 0:00:00
time: 13.9 s (started: 2026-03-29 12:32:27 +00:00)


In [79]:
# maybe don't do this for the larger dataset? Just assume SEG_1 is lung and SEG_2 is nodule?
def identify_and_rename_masks_using_pydicom(seg_file, timestep_path, prefix="SEG"):
  ds = pydicom.dcmread(seg_file)
  segment_labels = []
  for segment in ds.SegmentSequence:
    label = segment.SegmentLabel.lower().replace(" ", "_")
    segment_labels.append(label)
  for i, label in enumerate(segment_labels):
    segment_number = i+1
    old_name = os.path.join(timestep_path, f"{prefix}-{segment_number}.nrrd")
    new_name = os.path.join(timestep_path, f"{label}.nrrd")
    if os.path.exists(old_name):
      os.rename(old_name, new_name)
    else:
      print(f"File not found: {old_name}")

time: 1.43 ms (started: 2026-03-29 14:08:23 +00:00)


In [80]:
def identify_and_rename_masks(seg_file, timestep_path, prefix="SEG"):
  json_files = glob.glob(os.path.join(timestep_path, f"{prefix}-*.json"))
  if not json_files:
    identify_and_rename_masks_using_pydicom(seg_file, timestep_path, prefix)
  else:
    with open(json_files[0], 'r') as f:
      metadata = json.load(f)
      for i, segment in enumerate(metadata.get('segments', [])):
        label = segment.get('SegmentLabel', '').lower()
        segment_number = i+1
        old_name = os.path.join(timestep_path, f"{prefix}-{segment_number}.nrrd")
        if "lung" in label:
          new_name = os.path.join(timestep_path, "lung.nrrd")
          os.rename(old_name, new_name)
        elif "nodule" in label:
          new_name = os.path.join(timestep_path, "nodule.nrrd")
          os.rename(old_name, new_name)
    os.remove(json_files[0])

time: 2.28 ms (started: 2026-03-29 14:08:25 +00:00)


In [90]:
def identify_and_rename_masks_assuming_naming_convention(timestep_path, prefix="SEG"):
  os.rename(os.path.join(timestep_path, f"{prefix}-1.nrrd"), os.path.join(timestep_path, "lung.nrrd"))
  os.rename(os.path.join(timestep_path, f"{prefix}-2.nrrd"), os.path.join(timestep_path, "nodule.nrrd"))
  os.remove(os.path.join(timestep_path, f"{prefix}-meta.json"))

time: 1.89 ms (started: 2026-03-29 14:17:38 +00:00)


In [82]:
def align_masks(timestep_path):
  ct = sitk.ReadImage(os.path.join(timestep_path, "CT.nrrd"))
  lung_mask = sitk.ReadImage(os.path.join(timestep_path, "lung.nrrd"))
  nodule_mask = sitk.ReadImage(os.path.join(timestep_path, "nodule.nrrd"))
  resampler = sitk.ResampleImageFilter()
  resampler.SetReferenceImage(ct)
  resampler.SetInterpolator(sitk.sitkNearestNeighbor)
  aligned_lung_mask = resampler.Execute(lung_mask)
  aligned_nodule_mask = resampler.Execute(nodule_mask)
  sitk.WriteImage(aligned_lung_mask, os.path.join(timestep_path, "lung_aligned.nrrd"))
  sitk.WriteImage(aligned_nodule_mask, os.path.join(timestep_path, "nodule_aligned.nrrd"))

time: 1.47 ms (started: 2026-03-29 14:08:29 +00:00)


In [24]:
%pip install pynrrd
import nrrd
import numpy as np
import matplotlib.pyplot as plt

time: 5.2 s (started: 2026-03-29 12:47:44 +00:00)


In [83]:
def plot_ct_with_masks(patient_id, timestep_dir, timestep_path):
  ct, _ = nrrd.read(os.path.join(timestep_path, "CT.nrrd"))
  lung, _ = nrrd.read(os.path.join(timestep_path, "lung_aligned.nrrd"))
  nodule, _ = nrrd.read(os.path.join(timestep_path, "nodule_aligned.nrrd"))

  slice_counts = np.sum(nodule > 0, axis=(0,1))
  z_slice = np.argmax(slice_counts)
  ct_slice = ct[:, :, z_slice]
  lung_slice = lung[:, :, z_slice]
  nodule_slice = nodule[:, :, z_slice]

  fig, axes = plt.subplots(2,2,figsize=(12,12))
  axes[0,0].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=400, origin='lower')
  axes[0,0].set_title("Raw CT Slice")
  axes[0,0].axis('off')

  axes[0,1].imshow(ct_slice, cmap='gray', vmin=-1000, vmax=400, origin='lower')
  lung_mask = np.ma.masked_where(lung_slice == 0, lung_slice)
  axes[0,1].imshow(lung_mask, cmap='Blues', alpha=0.3, origin='lower', vmin=0, vmax=1)
  nodule_mask = np.ma.masked_where(nodule_slice == 0, nodule_slice)
  axes[0,1].imshow(nodule_mask, cmap='Reds', alpha=0.6, origin='lower', vmin=0, vmax=1)
  axes[0,1].set_title("CT Slice with Lung and Nodule Masks")
  axes[0,1].axis('off')

  axes[1,0].imshow(nodule_slice, cmap='gray', origin='lower')
  axes[1,0].set_title("Nodule Mask Only")
  axes[1,0].axis('off')

  nodule_voxels = ct_slice[nodule_slice > 0]
  axes[1,1].hist(nodule_voxels, bins=30, color='crimson', edgecolor='black')
  axes[1,1].set_title("Nodule HU Histogram")
  axes[1,1].set_xlabel("Hounsfield Units (HU)")
  axes[1,1].set_ylabel("Voxel Count")

  fig.suptitle(f"Patient ID: {patient_id}, Timestep: {timestep_dir}")
  plt.tight_layout()
  plt.savefig(os.path.join(timestep_path, "CT_with_masks.png"), dpi=300, bbox_inches='tight')
  plt.close(fig)

time: 2.27 ms (started: 2026-03-29 14:08:32 +00:00)


In [91]:
def prepare_for_pyradiomics_error_handling(base_path, align_and_plot_masks=True):
  for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    if not os.path.isdir(patient_path):
        continue

    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        if not os.path.isdir(timestep_path):
            continue

        ct_series_path = None
        seg_series_path = None

        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path

        if ct_series_path:
            try:
                convert_CT_to_nrrd(ct_series_path, timestep_path)
                print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {timestep_path}")
            except Exception as e:
                print(f"Failed to convert CT for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"CT series folder not found for patient {patient_id}, timestep {timestep_dir}. Skipping CT and SEG conversion for this timestep.")
            continue # Skip to next timestep if CT is not found

        if seg_series_path:
            try:
                seg_dcm_files = [f for f in os.listdir(seg_series_path) if f.endswith('.dcm')] # should be only one .dcm always, so remove this?
                if not seg_dcm_files:
                    print(f"No DICOM SEG files found in SEG series for patient {patient_id}, timestep {timestep_dir}")
                    continue
                seg_file = os.path.join(seg_series_path, seg_dcm_files[0])
                if convert_SEG_to_nrrd(seg_file, timestep_path):
                  print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
                  #identify_and_rename_masks(seg_file, timestep_path)
                  identify_and_rename_masks_assuming_naming_convention(timestep_path)
                  print(f"Renamed masks for patient {patient_id}, timestep {timestep_dir}")
                  if align_and_plot_masks:
                    align_masks(timestep_path)
                    print(f"Aligned masks for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
                    plot_ct_with_masks(patient_id, timestep_dir, timestep_path)
                    print(f"Plotted CT with masks for patient {patient_id}, timestep {timestep_dir}")
            except Exception as e:
                print(f"Failed to convert SEG for patient {patient_id}, timestep {timestep_dir}: {e}")
        else:
            print(f"SEG series folder not found for patient {patient_id}, timestep {timestep_dir}")

time: 5.06 ms (started: 2026-03-29 14:19:13 +00:00)


In [94]:
def prepare_for_pyradiomics(base_path, align_and_plot_masks=True):
  for patient_id in os.listdir(base_path):
    patient_path = os.path.join(base_path, patient_id)
    for timestep_dir in os.listdir(patient_path):
        timestep_path = os.path.join(patient_path, timestep_dir)
        for item in os.listdir(timestep_path):
            full_item_path = os.path.join(timestep_path, item)
            if os.path.isdir(full_item_path):
                if item.startswith('CT_'):
                    ct_series_path = full_item_path
                elif item.startswith('SEG_'):
                    seg_series_path = full_item_path
        try:
          convert_CT_to_nrrd(ct_series_path, timestep_path)
          print(f"Converted CT for patient {patient_id}, timestep {timestep_dir} to {timestep_path}")

          seg_dcm_files = [f for f in os.listdir(seg_series_path) if f.endswith('.dcm')] # should be only one .dcm always, so remove this?
          seg_file = os.path.join(seg_series_path, seg_dcm_files[0])
          convert_SEG_to_nrrd(seg_file, timestep_path)
          print(f"Converted SEG for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")

          identify_and_rename_masks_assuming_naming_convention(timestep_path)
          print(f"Renamed masks for patient {patient_id}, timestep {timestep_dir}")

          if align_and_plot_masks:
            align_masks(timestep_path)
            print(f"Aligned masks for patient {patient_id}, timestep {timestep_dir} to {timestep_dir}")
            plot_ct_with_masks(patient_id, timestep_dir, timestep_path)
            print(f"Plotted CT with masks for patient {patient_id}, timestep {timestep_dir}")

        except Exception as e:
                print(f"Failed for patient {patient_id}, timestep {timestep_dir}: {e}")

time: 8.72 ms (started: 2026-03-29 14:28:00 +00:00)


In [95]:
prepare_for_pyradiomics('/content/malignant_trial/nlst', align_and_plot_masks=False)

Converted CT for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352 to /content/malignant_trial/nlst/107030/1.2.840.113654.2.55.279742797608242185321251439412013425352
Converted SEG for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352 to 1.2.840.113654.2.55.279742797608242185321251439412013425352
Renamed masks for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352
Converted CT for patient 107030, timestep 1.2.840.113654.2.55.75126321162854904399604743122014916460 to /content/malignant_trial/nlst/107030/1.2.840.113654.2.55.75126321162854904399604743122014916460
Converted SEG for patient 107030, timestep 1.2.840.113654.2.55.75126321162854904399604743122014916460 to 1.2.840.113654.2.55.75126321162854904399604743122014916460
Renamed masks for patient 107030, timestep 1.2.840.113654.2.55.75126321162854904399604743122014916460
Converted CT for patient 126446, timestep 1.2.840.113654.2.55.2

In [96]:
prepare_for_pyradiomics('/content/malignant_trial/nlst')

Converted CT for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352 to /content/malignant_trial/nlst/107030/1.2.840.113654.2.55.279742797608242185321251439412013425352
Converted SEG for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352 to 1.2.840.113654.2.55.279742797608242185321251439412013425352
Renamed masks for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352
Aligned masks for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352 to 1.2.840.113654.2.55.279742797608242185321251439412013425352
Plotted CT with masks for patient 107030, timestep 1.2.840.113654.2.55.279742797608242185321251439412013425352
Converted CT for patient 107030, timestep 1.2.840.113654.2.55.75126321162854904399604743122014916460 to /content/malignant_trial/nlst/107030/1.2.840.113654.2.55.75126321162854904399604743122014916460
Converted SEG for patient 107030, timestep 1.2.840.1